[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skarma91/logicmojo-ai-july-2026/blob/main/modules/module-4-llms-genai/01-working-with-llms/code/logging_basics.ipynb)

# Class 4.1 companion: Python logging, properly

The main notebook used `logging` in passing. This one is a focused tour of the standard library `logging` module: levels, format, per-module loggers, exceptions, and the LLM-call habit. Everything runs offline. Use `force=True` on `basicConfig` so we can reconfigure it from cell to cell in a notebook.

## 1. Why not just print?

In [1]:
# print writes one undifferentiated stream: no severity, no timestamp, no off switch.
print("starting up")
print("something looks off")   # is this normal, a warning, or a crash? no way to tell

# logging tags every line with a level and a timestamp, and you can dial it up or down.
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", force=True)
logging.info("starting up")
logging.warning("something looks off")

2026-08-31 20:22:03,510 | INFO | starting up
2026-08-31 20:22:03,511 | WARNING | something looks off


starting up
something looks off


In [ ]:
# Configure the logging system with file

logging.basicConfig(
    filename='app.log', # Name of the file where the logs will be saved
    filemode='a',       # 'a' append mode, if you use 'w' then it will overwrite
    format="%(asctime)s | %(levelname)s | %(message)s",
    level=logging.INFO,
    force=True
)

# Test the logger
logging.debug("This is a debug message (useful for diagnosing problems)")
logging.info("This is an info message (confirming things work)")
logging.warning("This is a warning message (something unexpected happened)")
logging.error("This is an error message (a problem occurred)")
logging.critical("This is a critical message (the program may shut down)")

## 2. Levels: turn detail up or down

In [3]:
import logging
# Order of severity: DEBUG < INFO < WARNING < ERROR < CRITICAL.
# The level you set is the threshold; anything below it is suppressed.
logging.basicConfig(
    filename='app.log', # Name of the file where the logs will be saved
    filemode='a',       # 'a' append mode, if you use 'w' then it will overwrite
    format="%(asctime)s | %(levelname)s | %(message)s",
    level=logging.WARNING,
    force=True
)
logging.debug("noisy detail")     # suppressed at WARNING
logging.info("normal flow")       # suppressed at WARNING
logging.warning("heads up")       # shown
logging.error("this failed")      # shown
# Ship at INFO or WARNING; drop to DEBUG only while chasing a bug.

## 3. Format: add timestamps and context

In [5]:
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
logging.getLogger("app.py").info("now every line is timestamped and labeled")

20:34:02 | INFO    | app.py | now every line is timestamped and labeled


## 4. A logger per module

In [6]:
import logging
# Real projects call getLogger(__name__) in each module. Loggers form a hierarchy,
# so you can raise or lower detail for one component without touching the rest.
retriever_log = logging.getLogger("app.retriever")
llm_log = logging.getLogger("app.llm")

retriever_log.info("fetched 5 chunks")
llm_log.info("called the model")
# Later you could do logging.getLogger("app.retriever").setLevel(logging.DEBUG)
# to see retriever detail while everything else stays at INFO.

20:35:43 | INFO    | app.retriever | fetched 5 chunks
20:35:43 | INFO    | app.llm | called the model


## 5. Logging exceptions with the traceback

In [7]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s", force=True)
log = logging.getLogger("app")

try:
    1 / 0
except ZeroDivisionError:
    log.exception("computation failed")   # logs the message AND the full traceback
# log.exception is log.error plus the stack trace; call it inside an except block.

ERROR | computation failed
Traceback (most recent call last):
  File "C:\Users\Sourav Karmakar\AppData\Local\Temp\ipykernel_15696\3119305766.py", line 6, in <module>
    1 / 0
    ~~^~~
ZeroDivisionError: division by zero


## 6. The habit: log every LLM call

In [8]:
import logging, time
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s",
                    datefmt="%H:%M:%S", force=True)
log = logging.getLogger("llm")

def fake_call(prompt):
    time.sleep(0.05)
    return {"text": "(demo answer)", "in_tokens": 42, "out_tokens": 12, "cost": 0.0}

t0 = time.perf_counter()
out = fake_call("hello")
log.info("provider=%s in=%d out=%d cost=$%.6f latency=%.3fs",
         "echo", out["in_tokens"], out["out_tokens"], out["cost"], time.perf_counter() - t0)
# One INFO line per call, with everything you need to debug cost and speed later.

20:42:38 | INFO | provider=echo in=42 out=12 cost=$0.000000 latency=0.051s


In [11]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(filename)s | %(levelname)s | %(message)s",
                    datefmt="%H:%M:%S", force=True)

log = logging.getLogger(__name__)

log.info("This is just an info")

20:49:43 | 4029769921.py | INFO | This is just an info


## Recap

Logging gives you severity levels, timestamps, per-module control, and tracebacks, none of which `print` offers. In this course, log every model call at INFO with provider, tokens, cost, and latency. When an agent misbehaves in Module 5, that record is what makes it debuggable.